## Project: Auto-Tagging System for Content Categorization

### Task 1: Import Libraries and Modules

In [1]:
import spacy
import re
import pandas as pd
import contractions
import nltk
from spacy.matcher import Matcher
from spacy.training import Example
import random
from collections import Counter
import ast

### Task 2: Load and Explore the Dataset

In [2]:
# loading news data
df = pd.read_csv('/usercode/news_data.csv')

print(df.head())
print(df["News Data"].iloc[0])

                                           News Data
0  From: dlphknob@camelot.bradley.edu (Jemaleddin...
1  From: svoboda@rtsg.mot.com (David Svoboda)\nSu...
2  From: ld231782@longs.lance.colostate.edu (L. D...
3  From: ipser@solomon.technet.sg (Ed Ipser)\nSub...
4  From: cme@ellisun.sw.stratus.com (Carl Ellison...
From: dlphknob@camelot.bradley.edu (Jemaleddin Cole)
Subject: Re: Catholic Lit-Crit of a.s.s.
Nntp-Posting-Host: camelot.bradley.edu
Organization: The Society for the Preservation of Cruelty to Homophobes.
Lines: 37

In <1993Apr14.101241.476@mtechca.maintech.com> foster@mtechca.maintech.com writes:

>I am surprised and saddened. I would expect this kind of behavior
>from the Evangelical Born-Again Gospel-Thumping In-Your-Face We're-
>The-Only-True-Christian Protestants, but I have always thought 
>that Catholics behaved better than this.
>                                   Please do not stoop to the
>level of the E B-A G-T I-Y-F W-T-O-T-C Protestants, who think
>that the be

### Task 3: Handle Text Case, Contractions, and URLs

In [3]:
test_data = df["News Data"].iloc[0] 

In [4]:
def convert_to_lowercase(test_data): 
    return test_data.lower()

test_data = convert_to_lowercase(test_data)

print(test_data)

from: dlphknob@camelot.bradley.edu (jemaleddin cole)
subject: re: catholic lit-crit of a.s.s.
nntp-posting-host: camelot.bradley.edu
organization: the society for the preservation of cruelty to homophobes.
lines: 37

in <1993apr14.101241.476@mtechca.maintech.com> foster@mtechca.maintech.com writes:

>i am surprised and saddened. i would expect this kind of behavior
>from the evangelical born-again gospel-thumping in-your-face we're-
>the-only-true-christian protestants, but i have always thought 
>that catholics behaved better than this.
>                                   please do not stoop to the
>level of the e b-a g-t i-y-f w-t-o-t-c protestants, who think
>that the best way to witness is to be strident, intrusive, loud,
>insulting and overbearingly self-righteous.

(pleading mode on)

please!  i'm begging you!  quit confusing religious groups, and stop
making generalizations!  i'm a protestant!  i'm an evangelical!  i don't
believe that my way is the only way!  i'm not a "creatio

In [5]:
def expand_contractions(test_data):
    return contractions.fix(test_data)

test_data = expand_contractions(test_data)

print(test_data)

from: dlphknob@camelot.bradley.edu (jemaleddin cole)
subject: re: catholic lit-crit of a.s.s.
nntp-posting-host: camelot.bradley.edu
organization: the society for the preservation of cruelty to homophobes.
lines: 37

in <1993apr14.101241.476@mtechca.maintech.com> foster@mtechca.maintech.com writes:

>i am surprised and saddened. i would expect this kind of behavior
>from the evangelical born-again gospel-thumping in-your-face we are-
>the-only-true-christian protestants, but i have always thought 
>that catholics behaved better than this.
>                                   please do not stoop to the
>level of the e b-a g-t i-y-f w-t-o-t-c protestants, who think
>that the best way to witness is to be strident, intrusive, loud,
>insulting and overbearingly self-righteous.

(pleading mode on)

please!  i am begging you!  quit confusing religious groups, and stop
making generalizations!  i am a protestant!  i am an evangelical!  i do not
believe that my way is the only way!  i am not a "c

In [6]:
def remove_urls(test_data): 
    url_pattern = r'https?://\S+|www\\.\S+'
    return re.sub(url_pattern, ' ', test_data)

test_data = remove_urls(test_data)

print(test_data)

from: dlphknob@camelot.bradley.edu (jemaleddin cole)
subject: re: catholic lit-crit of a.s.s.
nntp-posting-host: camelot.bradley.edu
organization: the society for the preservation of cruelty to homophobes.
lines: 37

in <1993apr14.101241.476@mtechca.maintech.com> foster@mtechca.maintech.com writes:

>i am surprised and saddened. i would expect this kind of behavior
>from the evangelical born-again gospel-thumping in-your-face we are-
>the-only-true-christian protestants, but i have always thought 
>that catholics behaved better than this.
>                                   please do not stoop to the
>level of the e b-a g-t i-y-f w-t-o-t-c protestants, who think
>that the best way to witness is to be strident, intrusive, loud,
>insulting and overbearingly self-righteous.

(pleading mode on)

please!  i am begging you!  quit confusing religious groups, and stop
making generalizations!  i am a protestant!  i am an evangelical!  i do not
believe that my way is the only way!  i am not a "c

### Task 4: Handle Emails and Datetime

In [7]:
def remove_email_addresses(test_data):
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Z|a-z]{2,}\b'
    return re.sub(email_pattern, ' ', test_data)

test_data = remove_email_addresses(test_data)
test_data = re.sub(r'\s{2,}',' ', test_data).strip()

In [8]:
def remove_dates_times(test_data): 
    date_pattern = r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'
    time_pattern = r'\b\d{1,2}:\d{2}(?: \s?[AP]M)?\b' 
    test_data = re.sub(date_pattern, ' ', test_data) 
    test_data = re.sub(time_pattern, ' ', test_data) 
    return test_data

test_data = remove_dates_times(test_data)
test_data = re.sub(r'\s{2,}',' ', test_data).strip()

print(test_data)

from: dlphknob@camelot.bradley.edu (jemaleddin cole)
subject: re: catholic lit-crit of a.s.s.
nntp-posting-host: camelot.bradley.edu
organization: the society for the preservation of cruelty to homophobes.
lines: 37 in <1993apr14.101241.476@mtechca.maintech.com> foster@mtechca.maintech.com writes: >i am surprised and saddened. i would expect this kind of behavior
>from the evangelical born-again gospel-thumping in-your-face we are-
>the-only-true-christian protestants, but i have always thought >that catholics behaved better than this.
> please do not stoop to the
>level of the e b-a g-t i-y-f w-t-o-t-c protestants, who think
>that the best way to witness is to be strident, intrusive, loud,
>insulting and overbearingly self-righteous. (pleading mode on) please! i am begging you! quit confusing religious groups, and stop
making generalizations! i am a protestant! i am an evangelical! i do not
believe that my way is the only way! i am not a "creation scientist"! i
do not think that homos

### Task 5: Remove Numbers and Special Characters

In [9]:
def remove_numbers_and_special_characters(test_data):
    number_pattern = r'\b\d+\b'
    special_char_pattern = r'[^\w\s\\.]|_|\s+'
    test_data =  re.sub(number_pattern, ' ', test_data)
    test_data = re.sub(r'\s{2,}',' ', test_data).strip()
    return re.sub(special_char_pattern, ' ', test_data)

test_data = remove_numbers_and_special_characters(test_data)
test_data = re.sub(r'\s{2,}',' ', test_data).strip()

print(test_data)

from dlphknob camelot.bradley.edu jemaleddin cole subject re catholic lit crit of a.s.s. nntp posting host camelot.bradley.edu organization the society for the preservation of cruelty to homophobes. lines in 1993apr14. . mtechca.maintech.com foster mtechca.maintech.com writes i am surprised and saddened. i would expect this kind of behavior from the evangelical born again gospel thumping in your face we are the only true christian protestants but i have always thought that catholics behaved better than this. please do not stoop to the level of the e b a g t i y f w t o t c protestants who think that the best way to witness is to be strident intrusive loud insulting and overbearingly self righteous. pleading mode on please i am begging you quit confusing religious groups and stop making generalizations i am a protestant i am an evangelical i do not believe that my way is the only way i am not a creation scientist i do not think that homosexuals should be hung by their toenails if you wa

### Task 6: Handle Stopwords and Extra Spaces

In [10]:
nlp = spacy.load('en_core_web_md', disable=[ 'parser', 'lemmatizer', 'attribute_ruler'])

def remove_stop_words_and_spaces(test_data, nlp_model):
    doc = nlp_model(test_data)
    filtered_text = [token.text for token in doc if not token.is_stop]
    return(' '.join(filtered_text))

# Assuming nlp is defined elsewhere and passed appropriately
test_data = remove_stop_words_and_spaces(test_data, nlp)

print(test_data)

dlphknob camelot.bradley.edu jemaleddin cole subject catholic lit crit a.s.s . nntp posting host camelot.bradley.edu organization society preservation cruelty homophobes . lines 1993apr14 . . mtechca.maintech.com foster mtechca.maintech.com writes surprised saddened . expect kind behavior evangelical born gospel thumping face true christian protestants thought catholics behaved better . stoop level e b g t y f w t o t c protestants think best way witness strident intrusive loud insulting overbearingly self righteous . pleading mode begging quit confusing religious groups stop making generalizations protestant evangelical believe way way creation scientist think homosexuals hung toenails want discuss bible thumpers better singling making obtuse generalizations fundamentalists . compared actions presbyterians methodists southern baptists think different religions prejudice thinking people group write protestants evangelicals pleading mode . god ....... wish ahold thomas stories ...... fb

### Task 7: Tokenize Cleaned Text

In [11]:
def tokenize_text(text):
    tokens = [token.text for token in nlp(text)]
    return tokens

tokens = tokenize_text(test_data)

print(tokens)

['dlphknob', 'camelot.bradley.edu', 'jemaleddin', 'cole', 'subject', 'catholic', 'lit', 'crit', 'a.s.s', '.', 'nntp', 'posting', 'host', 'camelot.bradley.edu', 'organization', 'society', 'preservation', 'cruelty', 'homophobes', '.', 'lines', '1993apr14', '.', '.', 'mtechca.maintech.com', 'foster', 'mtechca.maintech.com', 'writes', 'surprised', 'saddened', '.', 'expect', 'kind', 'behavior', 'evangelical', 'born', 'gospel', 'thumping', 'face', 'true', 'christian', 'protestants', 'thought', 'catholics', 'behaved', 'better', '.', 'stoop', 'level', 'e', 'b', 'g', 't', 'y', 'f', 'w', 't', 'o', 't', 'c', 'protestants', 'think', 'best', 'way', 'witness', 'strident', 'intrusive', 'loud', 'insulting', 'overbearingly', 'self', 'righteous', '.', 'pleading', 'mode', 'begging', 'quit', 'confusing', 'religious', 'groups', 'stop', 'making', 'generalizations', 'protestant', 'evangelical', 'believe', 'way', 'way', 'creation', 'scientist', 'think', 'homosexuals', 'hung', 'toenails', 'want', 'discuss', 'b

### Task 8: Build Data Prep Pipeline

In [12]:
def preprocess_data(text):
    # Apply all preprocessing functions
    text = convert_to_lowercase(text)
    text = expand_contractions(text)
    text = remove_urls(text)
    text = remove_email_addresses(text)
    text = remove_dates_times(text)
    text = remove_numbers_and_special_characters(text)
    text = remove_stop_words_and_spaces(text, nlp)
    return text

In [13]:
def preprocess_and_tokenize(text):
    tokens = tokenize_text(text)
    return tokens

In [14]:
df['Processed_Data'] = df['News Data'].apply(preprocess_data)

df['Processed_Tokenized_Data'] = df['Processed_Data'].apply(preprocess_and_tokenize)

print(df.head())

                                           News Data  \
0  From: dlphknob@camelot.bradley.edu (Jemaleddin...   
1  From: svoboda@rtsg.mot.com (David Svoboda)\nSu...   
2  From: ld231782@longs.lance.colostate.edu (L. D...   
3  From: ipser@solomon.technet.sg (Ed Ipser)\nSub...   
4  From: cme@ellisun.sw.stratus.com (Carl Ellison...   

                                      Processed_Data  \
0    dlphknob camelot.bradley.edu   jemaleddin co...   
1    svoboda rtsg.mot.com   david svoboda   subje...   
2    ld231782 longs.lance.colostate.edu   l. detw...   
3    ipser solomon.technet.sg   ed ipser   subjec...   
4    cme ellisun.sw.stratus.com   carl ellison   ...   

                            Processed_Tokenized_Data  
0  [  , dlphknob, camelot.bradley.edu,   , jemale...  
1  [  , svoboda, rtsg.mot.com,   , david, svoboda...  
2  [  , ld231782, longs.lance.colostate.edu,   , ...  
3  [  , ipser, solomon.technet.sg,   , ed, ipser,...  
4  [  , cme, ellisun.sw.stratus.com,   , carl, el..

### Task 9: Create Pattern Matching Flow

In [15]:
# Define a function to find matches in a text
def find_matches(text, patterns):
    # Create a Matcher object
    matcher = Matcher(nlp.vocab)

    # Add patterns to the matcher object
    for pattern_name, pattern in patterns.items():
        matcher.add(pattern_name, [pattern])

    doc = nlp(text)
    matches = matcher(doc)
    matched_entities = []

    for match_id, start, end in matches:
        matched_entities.append((doc[start:end].text, nlp.vocab.strings[match_id]))

    return matched_entities

patterns = {
    "DATE_PATTERN": [
        {"IS_DIGIT": True, "LENGTH": 2, "OP": "?"},  # Day as two digits
        {"LOWER": {"IN": ["jan", "feb", "mar", "apr", "may", "jun", 
                          "jul", "aug", "sep", "oct", "nov", "dec"]}},  # Month abbreviations
        {"IS_DIGIT": True, "LENGTH": 4, "OP": "?"}  # Four digit year
    ],
    "MONEY_PATTERN": [
        {"ORTH": "$"},  # Optional dollar sign
        {"LIKE_NUM": True},  # Numeric value
        {"LOWER": {"IN": ["thousand", "million", "billion"]}, "OP": "?"}  # Large amount modifiers
    ]
}

sample_text = "I invested $1000 on 11 Jan 2021 and it grew to $5 million by Dec 2022."

matched_entities = find_matches(sample_text, patterns)

for entity, label in matched_entities:
    print(f"{entity} - {label}")

$1000 - MONEY_PATTERN
11 Jan - DATE_PATTERN
11 Jan 2021 - DATE_PATTERN
Jan - DATE_PATTERN
Jan 2021 - DATE_PATTERN
$5 - MONEY_PATTERN
$5 million - MONEY_PATTERN
Dec - DATE_PATTERN
Dec 2022 - DATE_PATTERN


### Task 10: Entity Extraction Using spaCy Model

In [16]:
# Take any sample processed text of your choice
sample_text = df["Processed_Data"].iloc[122]

def get_entities_medium_spacy(text):
    doc = nlp(text)
    entities = {}
    for ent in doc.ents:
        if ent.label_ not in entities:
            entities[ent.label_] = []
        entities[ent.label_].append(ent.text)
    return entities

entities = get_entities_medium_spacy(sample_text)

print(entities)

{'PERSON': ['tony jones', 'erik asphaug x2773', 'asphaug lpl.arizona.edu', 'tony', 'tony jones'], 'ORG': ['nntp', 'cray research inc   eagan   mn x', 'cmcs codegeneration group', 'cray research inc   '], 'NORP': ['pl6'], 'GPE': ['655f']}


### Task 11: Optimizing spaCy Model

In [17]:
# Annotated training data
train_data = [

     ("Noida is a city in India", {"entities": [(0, 5, "LOC")]}),
     ("Google launches a new AI research lab in Zurich", {"entities": [(0, 6, "ORG"), (37, 43, "LOC")]}),
     ("Amazon acquires Twitch for $970 million in 2014", {"entities": [(0, 6, "ORG"), (15, 21, "ORG"), (35, 42, "MONEY"), (46, 50, "DATE")]}),
     ("Elon Musk founded SpaceX to revolutionize space travel", {"entities": [(0, 9, "PERSON"), (18, 24, "ORG")]}),
     ("The Mona Lisa is on display in the Louvre Museum in Paris", {"entities": [(4, 13, "WORK_OF_ART"), (34, 47, "ORG"), (51, 56, "LOC")]}),
     ("The Great Wall of China stretches over 13,000 miles", {"entities": [(4, 22, "LOC"), (42, 51, "QUANTITY")]}),
     ("IBM introduces Watson, the AI that beat Jeopardy champions", {"entities": [(0, 3, "ORG"), (16, 22, "PERSON"), (31, 33, "ORG")]}),
     ("The Nile River flows through Egypt", {"entities": [(4, 14, "LOC"), (28, 33, "GPE")]}),
     ("Harvard University was established in 1636", {"entities": [(0, 17, "ORG"), (35, 39, "DATE")]}),
     ("Mount Everest is the world's highest mountain", {"entities": [(0, 13, "LOC")]}),
     ("Julia Roberts stars in the new Netflix series", {"entities": [(0, 12, "PERSON"), (31, 38, "ORG")]})
]

In [18]:
# Update the NER component with new examples
ner = nlp.get_pipe('ner')
for _, annotations in train_data:
   for ent in annotations.get('entities'):
      ner.add_label(ent[2])

# Disable other pipeline components for training
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
with nlp.disable_pipes(*other_pipes):
   optimizer = nlp.resume_training()
   for iteration in range(30):  # Adjust iterations as needed
      random.shuffle(train_data)
      for text, annotations in train_data:
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annotations)
            nlp.update([example], drop=0.5, sgd=optimizer)

/usr/local/lib/python3.8/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Julia Roberts stars in the new Netflix series" with entities "[(0, 12, 'PERSON'), (31, 38, 'ORG')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "The Great Wall of China stretches over 13,000 mile..." with entities "[(4, 22, 'LOC'), (42, 51, 'QUANTITY')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.8/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Harvard University was

In [19]:
entities = get_entities_medium_spacy("Noida is a city")
print(entities)

{'LOC': ['Noida']}


In [20]:
nlp.to_disk('/usercode/spacy_optimised')

### Task 12: Optimizing Entity Extraction for Auto-Tagging

In [21]:
def extract_entities_pipe(texts):
    entities = []
    for doc in nlp.pipe(texts):
        entities.append([(ent.text, ent.label_) for ent in doc.ents])
    return entities

if 'Processed_Data' in df.columns:
    df['entities'] = extract_entities_pipe(df['Processed_Data'])
    print(df.head())
else:
    print("The 'Processed_Data' column does not exist in the DataFrame.")

                                           News Data  \
0  From: dlphknob@camelot.bradley.edu (Jemaleddin...   
1  From: svoboda@rtsg.mot.com (David Svoboda)\nSu...   
2  From: ld231782@longs.lance.colostate.edu (L. D...   
3  From: ipser@solomon.technet.sg (Ed Ipser)\nSub...   
4  From: cme@ellisun.sw.stratus.com (Carl Ellison...   

                                      Processed_Data  \
0    dlphknob camelot.bradley.edu   jemaleddin co...   
1    svoboda rtsg.mot.com   david svoboda   subje...   
2    ld231782 longs.lance.colostate.edu   l. detw...   
3    ipser solomon.technet.sg   ed ipser   subjec...   
4    cme ellisun.sw.stratus.com   carl ellison   ...   

                            Processed_Tokenized_Data  \
0  [  , dlphknob, camelot.bradley.edu,   , jemale...   
1  [  , svoboda, rtsg.mot.com,   , david, svoboda...   
2  [  , ld231782, longs.lance.colostate.edu,   , ...   
3  [  , ipser, solomon.technet.sg,   , ed, ipser,...   
4  [  , cme, ellisun.sw.stratus.com,   , carl,

### Task 13: Enhanching Entity Aggregation for Workflow Optimization

In [22]:
def aggregate_entities(entities_list):
    aggregated_entities = {}
    for ent_text, ent_label in entities_list:
        if ent_label in aggregated_entities:
            aggregated_entities[ent_label].add(ent_text)
        else:
            aggregated_entities[ent_label] = {ent_text}
    return {label: list(texts) for label, texts in aggregated_entities.items()}

df['Refined_Entities'] = df['entities'].apply(aggregate_entities)

print(df["Refined_Entities"].head())

0    {'PRODUCT': ['cole'], 'ORG': ['nntp', 'thomas'...
1    {'PERSON': ['dave svoboda', 'beth   ', 'david ...
2    {'PERSON': ['richard e.', 'hoey zogwarg.etl.ar...
3    {'ORG': ['senate', 'solomon.technet.sg', 'step...
4    {'PERSON': ['carl ellison'], 'ORG': ['micmail'...
Name: Refined_Entities, dtype: object


### Task 14: Preparing and Refining Test Data for Entity Analysis

In [23]:
df_test = pd.read_csv('/usercode/test_data.csv')

# Fixing the Actual_Entities column
def dict_fix(val):
    return ast.literal_eval(val)

df_test["Actual_Entities"] = df_test["Actual_Entities"].apply(dict_fix)

df_test['Processed_Data'] = df_test['News Data'].apply(preprocess_data)

df_test['entities'] = extract_entities_pipe(df_test['Processed_Data'])

df_test['Refined_Entities'] = df_test['entities'].apply(aggregate_entities)

df_test.drop(["Processed_Data","entities"], axis=1, inplace=True)

df_test.head()

,News Data,Actual_Entities,Refined_Entities
0,From: rcollins@ns.encore.com (Roger Collins)\n...,"{'PERSON': ['roger collins', 'clinton', 'steve...","{'ORG': ['ns.encore.com', 'nntp', 'clinton', '..."
1,From: baalke@kelvin.jpl.nasa.gov (Ron Baalke)\...,"{'PERSON': ['daniel burstein', 'ron baalke', '...","{'PERSON': ['ron baalke', 'dannyb', 'daniel bu..."
2,Distribution: world\nFrom: David_A._Schnider@b...,"{'ORG': ['mac vga', 'sony cpd', 'cpd', 'trinit...","{'WORK_OF_ART': ['david ', 'david a. sch..."
3,From: brian@nostromo.NoSubdomain.NoDomain (Bri...,"{'PERSON': ['brian colaric sun', 'brian colari...","{'ORG': ['dse', 'os2']}"
4,From: fabian@vivian.w.open.de (Fabian Hoppe)\n...,"{'NORP': ['fabian'], 'ORG': ['nntp'], 'PERSON'...","{'PERSON': ['vivian.w.open.de '], 'ORG': ['n..."


### Task 15: Compute the Evaluation Metrics

In [24]:
def calculate_entity_metrics(actual, predicted):
    precision, recall, f1 = 0, 0, 0
    num_entities = len(actual)

    for ent_type in actual:
        actual_entities = set(actual.get(ent_type, []))
        predicted_entities = set(predicted.get(ent_type, []))

        true_positives = len(actual_entities & predicted_entities)
        false_positives = len(predicted_entities - actual_entities)
        false_negatives = len(actual_entities - predicted_entities)

        precision_part = true_positives / (true_positives + false_positives) if true_positives + false_positives > 0 else 0
        recall_part = true_positives / (true_positives + false_negatives) if true_positives + false_negatives > 0 else 0
        
        precision += precision_part
        recall += recall_part

     # Handle the case where there are no entities
    if num_entities > 0:
        precision /= num_entities
        recall /= num_entities
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
    else:
        precision, recall, f1 = 0, 0, 0  

    return precision, recall, f1


metrics = df_test.apply(lambda row: calculate_entity_metrics(row['Actual_Entities'], row['Refined_Entities']), axis=1)

In [25]:
precisions, recalls, f1s = zip(*metrics)
average_precision = sum(precisions) / len(precisions)
average_recall = sum(recalls) / len(recalls)
average_f1 = sum(f1s) / len(f1s)

print(f'Average Precision: {average_precision}')
print(f'Average Recall: {average_recall}')
print(f'Average f1: {average_f1}')

Average Precision: 0.2598355590444926
Average Recall: 0.17738930374577044
Average f1: 0.20004445426824627
